In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
OUT_DIR = "./outputs_localsoil"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import existing implementations
from models.train_model import get_model
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")

    print(f"Loading data from {npz_path}...")
    d = np.load(npz_path, allow_pickle=True)

    # 1. Load exactly the four arrays (fail if keys missing)
    try:
        X_train = d["X_train"].astype(float)
        X_test  = d["X_test"].astype(float)
        y_train = d["y_train"].astype(int).ravel() # Ensure 1D
        y_test  = d["y_test"].astype(int).ravel()
    except KeyError as e:
        raise KeyError(f"Missing expected key in NPZ: {e}. Keys found: {list(d.keys())}")

    # 2. Load feature names if available, else default
    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    # 3. Strict Shape Assertions (Fail Fast)
    print(f"Shapes -> X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape}")

    assert X_train.ndim == 2, "X_train must be 2D"
    assert X_test.ndim == 2,  "X_test must be 2D"
    assert X_train.shape[0] == y_train.shape[0], f"Mismatch: X_train rows {X_train.shape[0]} != y_train {y_train.shape[0]}"
    # But for a standard ML setup, they must match.
    assert X_test.shape[0] == y_test.shape[0], f"Mismatch: X_test rows {X_test.shape[0]} != y_test {y_test.shape[0]}"
    assert len(feature_names) == X_train.shape[1], "Mismatch between feature_names length and X columns"

    return X_train, y_train, X_test, y_test, feature_names

# Load Data
X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
n_features = X_train.shape[1]

# Ensure labels are a sorted list for consistent log_loss usage
all_labels = sorted(np.unique(y_train).tolist())
print(f"Labels detected: {all_labels}")

# Baseline: Mean of training data
baseline_full = np.mean(X_train, axis=0)

trained_models = {}
MODEL_NAMES = ["logistic", "svm", "random_forest"]

print("\nTraining models...")
for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)

    # Safety check: Ensure it is a classifier
    if not is_classifier(model):
         # If get_model returns a regressor but data implies classification, warn/error
         print(f"Warning: {m_name} does not seem to be a classifier (is_classifier=False).")

    model.fit(X_train, y_train)
    trained_models[m_name] = model

print("Models trained:", list(trained_models.keys()))

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true) # Ensure int
        self.all_labels = all_labels

        # Setup GT player interface with REAL feature names
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=[feature_names[j] for j in feature_subset]
        )

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        # Check if model supports probability
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            # Fallback for models like SVM (if probability=False) or others
            # Warning: log_loss is not strictly defined without probs,
            # but we can use 0-1 loss or raise error.
            # Here we try to interpret decision function or hard predict.
            pred = self.model.predict(X_in)[0]
            # Simple 0-1 Loss fallback
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        X_in = self._x_for_coalition(S)
        return -self._loss_value(X_in)

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("shapley_shubik", exact_shapley_shubik),
    ("banzhaf", exact_banzhaf_index),
]

records = []

print(f"\nRunning explanation on {len(test_idx)} samples using {len(METHODS)} methods...")

for model_name, model in trained_models.items():
    # Check if this model might fail log_loss (e.g. SVM without probability=True)
    if model_name == "svm" and not getattr(model, "probability", True) and not hasattr(model, "predict_proba"):
        print(f"Warning: SVM model might lack predict_proba. Fallback 0-1 loss will be used.")

    for i, idx in enumerate(test_idx):
        x = X_test[idx]
        y = y_test[idx]

        # Use ALL features (exact computation)
        feat_subset = np.arange(n_features)

        util = LocalLossUtility(
            model=model,
            x_full=x,
            y_true=y,
            baseline_full=baseline_full,
            feature_subset=feat_subset,
            feature_names=feature_names,
            all_labels=all_labels
        )

        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                # Create named dictionary for clarity
                alloc_named = {feature_names[j]: float(alloc[j]) for j in range(n_features)}

                rec = {
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y),
                    "method": method_name,
                    "alloc_full": alloc.tolist(),
                    "alloc_named": alloc_named
                }
                records.append(rec)
            except Exception as e:
                print(f"Error in {method_name} (Model: {model_name}, Idx: {idx}): {e}")

df = pd.DataFrame(records)
print("\nExperiment Complete. First 5 rows:")
print(df.head())

out_file = os.path.join(OUT_DIR, "loss_utility_results.csv")
df.to_csv(out_file, index=False)
print(f"\nResults saved to: {out_file}")

Loading data from ../../datasets/soil_spi_preprocessed.npz...
Shapes -> X_train: (4000, 7), y_train: (4000,), X_test: (1000, 7)
Labels detected: [0, 1, 2]

Training models...
Models trained: ['logistic', 'svm', 'random_forest']

Running explanation on 50 samples using 6 methods...

Experiment Complete. First 5 rows:
      model  test_index  y_true          method  \
0  logistic         210       0         shapley   
1  logistic         210       0      least_core   
2  logistic         210       0       nucleolus   
3  logistic         210       0          gately   
4  logistic         210       0  shapley_shubik   

                                          alloc_full  \
0  [0.4332037222255711, 0.7544623374476076, 0.590...   
1  [-0.01580790443075716, 0.00933958524815701, -0...   
2  [-0.014365911691893144, -0.0007680868699476388...   
3  [-0.027913659337746477, -0.004956302414021785,...   
4  [0.16666666666666666, 0.16666666666666666, 0.1...   

                                      

In [2]:
import os
import sys
import warnings
import logging
import json
from pathlib import Path
from typing import Dict, List, Callable, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

# Setup Paths
sys.path.append(os.path.abspath(".."))

# Import GT methods
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index
from models.train_model import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_localsoil/loss_utility_results.csv"
OUT_DIR = Path("./output_soil/quantus_evaluation_soil")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

# 1. utilities

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[i]) for i in self.feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        from sklearn.metrics import log_loss
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1: out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()
    return X_train, X_test, y_train, y_test

# 2. robust perturbation & distance functions

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    """
    Safely perturbs tabular data by replacing with baseline.
    """
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    if arr.ndim == 2:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b], idx_list)

    elif arr.ndim == 3:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get('lower_bound', 0.01)
    upper = kwargs.get('upper_bound', 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))


def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,  # Faithfulness uses subset_size
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,   # Selectivity uses patch_size
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1, # Monotonicity uses features_in_step
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1, # PixelFlipping uses features_in_step
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),

        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),

        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=5, perturb_std=0.1,
            norm_numerator=distance_euclidean,
            norm_denominator=distance_euclidean,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=5, lower_bound=0.1, upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        base_scores = base_probs[np.arange(len(y_test)), y_test]
    else:
        return np.nan

    drops_all_k = []
    for k in top_k_list:
        if k > X_test_2d.shape[1]: continue
        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]
        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]
        drops_all_k.append(np.mean(base_scores - new_scores))
    return np.mean(drops_all_k)

# 4. explain function

def make_explain_func(method_name, all_labels, baseline_full, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "nucleolus": exact_nucleolus,
        "gately": exact_gately_value,
        "shapley_shubik": exact_shapley_shubik,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        if isinstance(model, SklearnTorchWrapper):
            real_model = model.model
        else:
            real_model = model

        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = y_in.item() if hasattr(y_in, 'item') else y_in

            util = LocalLossUtility(
                model=real_model, x_full=x, y_true=y,
                baseline_full=baseline_full, feature_subset=np.arange(len(x)),
                feature_names=feature_names, all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception as e:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    X_train, X_test, y_train, y_test = load_data_strict()
    baseline = np.mean(X_train, axis=0)

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset = y_test[valid_indices]

    # Quantus Input: (N, 1, F)
    x_batch = X_subset[:, np.newaxis, :]

    models_to_run = df_results["model"].unique()
    methods_to_run = df_results["method"].unique()

    all_eval_rows = []
    metrics = get_metrics(baseline)

    feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    logger.info(f"Loaded {len(valid_indices)} samples for evaluation.")

    for model_name in models_to_run:
        logger.info(f"--- Evaluating Model: {model_name} ---")

        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()
        all_labels = sorted(np.unique(y_train).tolist())

        df_model = df_results[df_results["model"] == model_name]

        for method_name in methods_to_run:
            logger.info(f"   Method: {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if row.empty:
                    attributions_list.append(np.zeros(X_train.shape[1]))
                else:
                    val_str = row.iloc[0]["alloc_full"]
                    if isinstance(val_str, str):
                        val_arr = np.array(json.loads(val_str))
                    else:
                        val_arr = np.array(val_str)
                    attributions_list.append(val_arr)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}
            y_faith = y_subset if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset)

            # Global Metrics
            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_faith,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            # Aopc
            try:
                aopc = compute_aopc_manual(sk_model, X_subset, y_subset, a_batch_2d, AOPC_TOPK_LIST, baseline)
                row_stats["aopc_manual"] = aopc
            except Exception as e:
                 row_stats["aopc_manual"] = np.nan

            # Robustness
            explain_func = make_explain_func(method_name, all_labels, baseline, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                try:
                    scores = metrics[m_name](
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset,
                        explain_func=explain_func,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            all_eval_rows.append(row_stats)
            f_corr = row_stats.get('faithfulness_corr', np.nan)
            ms = row_stats.get('max_sensitivity', np.nan)
            logger.info(f"     -> Faith_Corr: {f_corr:.4f}, MaxSens: {ms:.4f}")

    df_eval = pd.DataFrame(all_eval_rows)
    out_path = OUT_DIR / "quantussoil_results_full_extended.csv"
    df_eval.to_csv(out_path, index=False)
    print(df_eval.to_string())

if __name__ == "__main__":
    main()

            model          method  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual  local_lipschitz_estimate  max_sensitivity
0        logistic         shapley           0.821429               0.797687     0.203708          0.44        0.203708    0.482146    1.491382     0.442269                  0.430822         0.566439
1        logistic      least_core           0.810747               0.836378     0.207823          0.36        0.207823    0.458666    1.546588     0.299024                  0.554232         1.164727
2        logistic       nucleolus           0.888345               0.861762     0.214939          0.38        0.214939    0.424505    1.582988     0.287040                  0.569848         1.370556
3        logistic          gately           0.879901               0.894976     0.216727          0.38        0.216727    0.400242    1.626086     0.303459                  0.654425         1.451075
4    

In [3]:
import os
import sys
import warnings
import logging
import ast
from pathlib import Path
from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import quantus

# Setup Paths
sys.path.append(os.path.abspath(".."))

# Import GT methods
from gt_concepts.shapley import exact_shapley_value
from gt_concepts.least_core import exact_least_core
from gt_concepts.nucleolus import exact_nucleolus
from gt_concepts.gately_point import exact_gately_value
from gt_concepts.shapley_shubik import exact_shapley_shubik
from gt_concepts.banzhaf import exact_banzhaf_index
from models.train_model import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Config
DATA_NPZ_PATH = "../../datasets/soil_spi_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_localsoil/loss_utility_results.csv"
OUT_DIR = Path("./outputs_soil/quantus_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
# Soil dataset has 7 features
AOPC_TOPK_LIST = [1, 2, 3, 4, 5, 6, 7]

# 1. utilities

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class LocalLossUtility:
    def __init__(self, model, x_full, y_true, baseline_full, feature_subset, feature_names, all_labels):
        self.model = model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.baseline_full = np.asarray(baseline_full, dtype=float).ravel()
        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.y_true = int(y_true)
        self.all_labels = all_labels

        subset_names = [str(feature_names[i]) for i in self.feature_subset]
        self.data = FeaturePlayers(indices=list(range(len(feature_subset))), data_names=subset_names)

    def _x_for_coalition(self, S):
        x = self.baseline_full.copy()
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            x[global_j] = self.x_full[global_j]
        return x.reshape(1, -1)

    def _loss_value(self, X_in) -> float:
        from sklearn.metrics import log_loss
        if hasattr(self.model, "predict_proba"):
            proba = self.model.predict_proba(X_in)
            return float(log_loss([self.y_true], proba, labels=self.all_labels))
        else:
            pred = self.model.predict(X_in)[0]
            return 0.0 if pred == self.y_true else 1.0

    def __call__(self, S):
        return -self._loss_value(self._x_for_coalition(S))

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        # Flatten (N, 1, F) -> (N, F) or (N, F) -> (N, F)
        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1: out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    return X_train, X_test, y_train, y_test, feature_names

# 2. robust perturbation

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    B = arr.shape[0]
    for b in range(B):
        idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
        if arr.ndim == 2:
            apply_mask(arr_pert[b], idx_list)
        elif arr.ndim == 3:
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get('lower_bound', 0.01)
    upper = kwargs.get('upper_bound', 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# 3. metrics

def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),

        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),

        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=5, perturb_std=0.1,
            norm_numerator=distance_euclidean,
            norm_denominator=distance_euclidean,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=5, lower_bound=0.1, upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        base_scores = base_probs[np.arange(len(y_test)), y_test]
    else:
        return np.nan

    drops_all_k = []
    # Ensure K doesn't exceed 7 (n_features)
    valid_k_list = [k for k in top_k_list if k <= X_test_2d.shape[1]]

    for k in valid_k_list:
        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]
        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]
        drops_all_k.append(np.mean(base_scores - new_scores))
    return np.mean(drops_all_k) if drops_all_k else np.nan

# 4. explain function

def make_explain_func(method_name, all_labels, baseline_full, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "nucleolus": exact_nucleolus,
        "gately": exact_gately_value,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        if isinstance(model, SklearnTorchWrapper):
            real_model = model.model
        else:
            real_model = model

        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = y_in.item() if hasattr(y_in, 'item') else y_in

            util = LocalLossUtility(
                model=real_model, x_full=x, y_true=y,
                baseline_full=baseline_full, feature_subset=np.arange(len(x)),
                feature_names=feature_names, all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception as e:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    if not os.path.exists(RESULTS_CSV_PATH):
        print(f"Error: Results file not found at {RESULTS_CSV_PATH}. Run script 1 first.")
        return

    X_train, X_test, y_train, y_test, feature_names = load_data_strict()
    baseline = np.mean(X_train, axis=0)
    all_labels = sorted(np.unique(y_train).tolist())

    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset = y_test[valid_indices]

    x_batch = X_subset[:, np.newaxis, :]

    models_to_run = df_results["model"].unique()
    methods_to_run = df_results["method"].unique()

    all_eval_rows = []
    metrics = get_metrics(baseline)

    logger.info(f"Loaded {len(valid_indices)} samples for evaluation.")

    for model_name in models_to_run:
        logger.info(f"--- Evaluating Model: {model_name} ---")

        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        torch_model = SklearnTorchWrapper(sk_model).eval()

        df_model = df_results[df_results["model"] == model_name]

        for method_name in methods_to_run:
            logger.info(f"   Method: {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if row.empty:
                    attributions_list.append(np.zeros(X_train.shape[1]))
                else:
                    val_str = row.iloc[0]["alloc_full"]
                    try:
                        if isinstance(val_str, str):
                            val_arr = np.array(ast.literal_eval(val_str))
                        else:
                            val_arr = np.array(val_str)
                    except Exception:
                        val_arr = np.zeros(X_train.shape[1])
                    attributions_list.append(val_arr)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}
            y_faith = y_subset if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset)

            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_faith,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            try:
                aopc = compute_aopc_manual(sk_model, X_subset, y_subset, a_batch_2d, AOPC_TOPK_LIST, baseline)
                row_stats["aopc_manual"] = aopc
            except Exception as e:
                 row_stats["aopc_manual"] = np.nan

            explain_func = make_explain_func(method_name, all_labels, baseline, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                try:
                    scores = metrics[m_name](
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset,
                        explain_func=explain_func,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            all_eval_rows.append(row_stats)
            f_corr = row_stats.get('faithfulness_corr', np.nan)
            ms = row_stats.get('max_sensitivity', np.nan)
            logger.info(f"     -> Faith_Corr: {f_corr:.4f}, MaxSens: {ms:.4f}")

    df_eval = pd.DataFrame(all_eval_rows)
    out_path = OUT_DIR / "quantus_results_soil.csv"
    df_eval.to_csv(out_path, index=False)
    print(df_eval.to_string())

if __name__ == "__main__":
    main()

            model          method  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual  local_lipschitz_estimate  max_sensitivity
0        logistic         shapley           0.846266               0.797687     0.203708          0.44        0.203708    0.482146    1.491382     0.442269                  0.419959         0.574997
1        logistic      least_core           0.835995               0.836378     0.207823          0.36        0.207823    0.458666    1.546588     0.299024                  0.558155         1.266122
2        logistic       nucleolus           0.888639               0.861762     0.214939          0.38        0.214939    0.424505    1.582988     0.287040                  0.642714         1.268549
3        logistic          gately           0.926480               0.894976     0.216727          0.38        0.216727    0.400242    1.626086     0.303459                  1.035421         1.399692
4    